# Prova di interoperabilità con HL7 FHIR — MIMIC-IV
### Tesi: *Progettazione di un modulo di interoperabilità per la sorveglianza automatizzata in terapia sub-intensiva*

**Domanda di questa prova:** se al modello arrivano gli stessi dati clinici in formato HL7 FHIR invece che nel formato originale di MIMIC-IV, ottengo le stesse 20 variabili e le stesse probabilità di deterioramento?

**Dati:** gli stessi 100 pazienti della demo pubblica di MIMIC-IV (v2.2), disponibili in due formati:
- **Strada A** — i file CSV originali (`mimic-iv-demo-csv/`), letti con la stessa logica del notebook `modello-MIMIC-corretto.ipynb`
- **Strada B** — la versione FHIR ufficiale (`mimic-iv-fhir-demo/`, progetto *MIMIC-IV on FHIR*), letta con un lettore FHIR che traduce ogni codice nel codice standard **LOINC** e poi nella variabile del modello

Dalle 20 variabili in poi le due strade usano **la stessa funzione** (ricampionamento, imputazione, NEWS-2, Delta-NEWS) e **lo stesso modello già addestrato** (`models/modello_news2_mimic.pkl`, nessun nuovo addestramento).

**Cosa NON misura:** le prestazioni del modello. 100 pazienti sono pochi e alcuni potrebbero essere tra quelli usati per l'addestramento: qui si verifica solo che le due strade diano lo stesso risultato (test di equivalenza).

Da eseguire con l'ambiente Python `esame`, come il notebook MIMIC-IV.

## PARTE 0 — Configurazione
- Percorsi delle due versioni della demo e dei modelli salvati
- Gli stessi codici (itemid) e le stesse soglie del notebook MIMIC-IV

In [1]:
import pandas as pd
import numpy as np
import gzip, json, joblib

CARTELLA_CSV  = "mimic-iv-demo-csv/mimic-iv-clinical-database-demo-2.2/"                        # strada A
CARTELLA_FHIR = "mimic-iv-fhir-demo/mimic-iv-clinical-database-demo-on-fhir-2.1.0/fhir/"        # strada B

# Stessi codici del notebook MIMIC-IV (usati SOLO dalla strada A)
ITEM_IDS = {
    220210: "freq_respiratoria", 220277: "spo2",
    220179: "pressione_sistolica", 220050: "pressione_sistolica",
    220045: "freq_cardiaca", 223762: "temperatura", 223900: "coscienza_gcs",
}
ID_TEMP_F = 223761
ID_O2_DISPOSITIVO, ID_O2_FIO2, ID_O2_FLUSSO = 226732, 223835, {223834, 227287}
O2_ITEM_IDS = {ID_O2_DISPOSITIVO, ID_O2_FIO2} | ID_O2_FLUSSO

# Le 20 feature del modello, nello stesso ordine usato in addestramento
FEATURES = [
    "freq_respiratoria", "spo2", "pressione_sistolica", "freq_cardiaca", "temperatura",
    "ossigeno_supplementare", "news2",
    "news2_max_6h", "news2_min_6h", "news2_media_6h", "news2_std_6h",
    "delta_news", "delta_news_6h", "n_parametri_mancanti",
    "freq_respiratoria_mancante", "spo2_mancante", "pressione_sistolica_mancante",
    "freq_cardiaca_mancante", "temperatura_mancante", "coscienza_gcs_mancante",
]

## PARTE 1 — Lo strato di interoperabilità: la tabella di traduzione
Questa tabella è il "modulo di interoperabilità" in forma concreta. Ha due passaggi:
1. **codice della sorgente → codice LOINC** (standard internazionale per le misure cliniche). La versione FHIR di MIMIC-IV codifica i parametri vitali di terapia intensiva con i codici interni di MIMIC, quindi questa traduzione la facciamo noi.
2. **codice LOINC → variabile del modello**. Questa parte non dipende dalla sorgente: qualunque ospedale che invia dati FHIR con codici LOINC può essere letto senza cambiare nulla.

I nomi LOINC sono stati verificati sul catalogo ufficiale (servizio *Clinical Tables* della National Library of Medicine, 26 settembre 2026).

In [2]:
# Passaggio 1: codice MIMIC (come compare nel FHIR) -> codice LOINC
SISTEMA_MIMIC = "http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-chartevents-d-items"
SISTEMA_LOINC = "http://loinc.org"
MIMIC_A_LOINC = {
    "220045": "8867-4",    # Heart rate
    "220210": "9279-1",    # Respiratory rate
    "220277": "59408-5",   # Oxygen saturation in Arterial blood by Pulse oximetry
    "220179": "8480-6",    # Systolic blood pressure (non invasiva)
    "220050": "76215-3",   # Invasive Systolic blood pressure
    "223762": "8310-5",    # Body temperature (in °C)
    "223761": "8310-5",    # Body temperature (in °F: l'unità viene dal campo "unit")
    "223900": "9270-0",    # Glasgow coma score verbal
    "226732": "107117-4",  # Method of oxygen delivery
    "223835": "3150-0",    # Inhaled oxygen concentration (FiO2)
    "223834": "3151-8",    # Inhaled oxygen flow rate
    "227287": "3151-8",    # Inhaled oxygen flow rate (cannula aggiuntiva)
}

# Passaggio 2: codice LOINC -> variabile del modello (indipendente dalla sorgente)
LOINC_A_VARIABILE = {
    "8867-4": "freq_cardiaca", "9279-1": "freq_respiratoria", "59408-5": "spo2",
    "8480-6": "pressione_sistolica", "76215-3": "pressione_sistolica",
    "8310-5": "temperatura", "9270-0": "coscienza_gcs",
}
LOINC_O2_DISPOSITIVO, LOINC_O2_FIO2, LOINC_O2_FLUSSO = "107117-4", "3150-0", "3151-8"

pd.DataFrame(
    [(m, l, LOINC_A_VARIABILE.get(l, "ossigeno_supplementare")) for m, l in MIMIC_A_LOINC.items()],
    columns=["codice MIMIC", "codice LOINC", "variabile del modello"])

,codice MIMIC,codice LOINC,variabile del modello
0,220045,8867-4,freq_cardiaca
1,220210,9279-1,freq_respiratoria
2,220277,59408-5,spo2
3,220179,8480-6,pressione_sistolica
4,220050,76215-3,pressione_sistolica
5,223762,8310-5,temperatura
6,223761,8310-5,temperatura
7,223900,9270-0,coscienza_gcs
8,226732,107117-4,ossigeno_supplementare
9,223835,3150-0,ossigeno_supplementare


## PARTE 2 — La parte comune alle due strade
Una sola funzione trasforma i dati "in formato comune" nelle 20 feature. Riceve due tabelle:
- **vitali**: una riga per misura, con `subject_id`, `stay_id`, `charttime`, `parametro`, `valuenum` (già in °C)
- **ossigeno**: una riga per registrazione, con `_o2` = 1 (ossigeno) o 0 (nessun dispositivo)

Il codice è copiato dal notebook MIMIC-IV (pivot, persistenza dell'ossigeno di 4 ore, pulizia, ricampionamento, imputazione con limite di 4 ore, NEWS-2, Delta-NEWS). Le due strade differiscono **solo** in come costruiscono queste due tabelle.

Nota tecnica: i valori entrano come `float32`, lo stesso tipo di dato con cui il notebook MIMIC-IV legge `valuenum`. Fa parte del formato comune: senza, un valore sul confine di una fascia del NEWS-2 (per esempio 96,8 °F = 36,0 °C) potrebbe cadere dalla parte opposta per un arrotondamento all'ultima cifra decimale.

In [3]:
def calcola_news2(row):
    # identica al notebook MIMIC-IV (coscienza = GCS verbale, soglia < 5)
    score = 0
    fr = row["freq_respiratoria"]
    if   fr <= 8:  score += 3
    elif fr <= 11: score += 1
    elif fr <= 20: score += 0
    elif fr <= 24: score += 2
    else:          score += 3
    spo2 = row["spo2"]
    if   spo2 <= 91: score += 3
    elif spo2 <= 93: score += 2
    elif spo2 <= 95: score += 1
    if row["ossigeno_supplementare"] > 0:
        score += 2
    pa = row["pressione_sistolica"]
    if   pa <= 90:  score += 3
    elif pa <= 100: score += 2
    elif pa <= 110: score += 1
    elif pa <= 219: score += 0
    else:           score += 3
    fc = row["freq_cardiaca"]
    if   fc <= 40:  score += 3
    elif fc <= 50:  score += 1
    elif fc <= 90:  score += 0
    elif fc <= 110: score += 1
    elif fc <= 130: score += 2
    else:           score += 3
    if row["coscienza_gcs"] < 5:
        score += 3
    temp = row["temperatura"]
    if   temp <= 35.0: score += 3
    elif temp <= 36.0: score += 1
    elif temp <= 38.0: score += 0
    elif temp <= 39.0: score += 1
    else:              score += 2
    return score


def features_temporali(gruppo):
    gruppo = gruppo.sort_index()
    rolling = gruppo["news2"].rolling("360min", min_periods=1)
    gruppo["news2_max_6h"]   = rolling.max()
    gruppo["news2_min_6h"]   = rolling.min()
    gruppo["news2_media_6h"] = rolling.mean()
    gruppo["news2_std_6h"]   = rolling.std().fillna(0)
    gruppo["delta_news"] = gruppo["news2"].diff().fillna(0)
    news_6h_fa = gruppo["news2"].rolling("360min", min_periods=2).apply(lambda x: x.iloc[0], raw=False)
    gruppo["delta_news_6h"] = (gruppo["news2"] - news_6h_fa).fillna(0)
    cols_mancanti = [c for c in gruppo.columns if c.endswith("_mancante")]
    gruppo["n_parametri_mancanti"] = gruppo[cols_mancanti].sum(axis=1)
    return gruppo


def pipeline_20_feature(vitali, ossigeno):
    # Dal formato comune alle 20 feature: stessi passi del notebook MIMIC-IV, dal pivot in poi
    vitali = vitali.copy()
    vitali["valuenum"] = vitali["valuenum"].astype("float32")
    vitali["charttime"] = pd.to_datetime(vitali["charttime"])
    ossigeno = ossigeno.copy()
    ossigeno["charttime"] = pd.to_datetime(ossigeno["charttime"])

    # Pivot (media se due fonti misurano lo stesso istante, es. pressione invasiva e non invasiva)
    df = vitali.pivot_table(index=["subject_id", "stay_id", "charttime"], columns="parametro",
                            values="valuenum", aggfunc="mean").reset_index()
    df.columns.name = None

    # Ossigeno: ultima registrazione dello stesso ricovero entro 4 ore
    ossigeno = ossigeno.groupby(["subject_id", "stay_id", "charttime"], as_index=False)["_o2"].max()
    ossigeno = ossigeno.sort_values("charttime")
    df = df.sort_values("charttime").reset_index(drop=True)
    stato = pd.merge_asof(df[["subject_id", "stay_id", "charttime"]], ossigeno, on="charttime",
                          by=["subject_id", "stay_id"], direction="backward", tolerance=pd.Timedelta("4h"))
    df["ossigeno_supplementare"] = stato["_o2"].fillna(0).astype(int).values

    # Pulizia: range fisiologici e salti impossibili
    filtri = {"freq_respiratoria": (4, 60), "spo2": (50, 100), "pressione_sistolica": (40, 300),
              "freq_cardiaca": (10, 300), "temperatura": (25, 45), "coscienza_gcs": (1, 5)}
    for colonna, (minimo, massimo) in filtri.items():
        if colonna in df.columns:
            df.loc[(df[colonna] < minimo) | (df[colonna] > massimo), colonna] = np.nan
    df = df.sort_values(["subject_id", "stay_id", "charttime"])
    df["delta_tempo_min"] = df.groupby(["subject_id", "stay_id"])["charttime"].diff().dt.total_seconds() / 60
    delta_fc = df.groupby(["subject_id", "stay_id"])["freq_cardiaca"].diff().abs()
    df.loc[(delta_fc > 50) & (df["delta_tempo_min"] <= 10), "freq_cardiaca"] = np.nan
    delta_pa = df.groupby(["subject_id", "stay_id"])["pressione_sistolica"].diff().abs()
    df.loc[(delta_pa > 60) & (df["delta_tempo_min"] <= 10), "pressione_sistolica"] = np.nan
    df = df.drop(columns=["delta_tempo_min"])

    # Ricampionamento a 10 minuti sulle sole misure vere
    parametri = ["freq_respiratoria", "spo2", "pressione_sistolica",
                 "freq_cardiaca", "temperatura", "coscienza_gcs"]
    df = df.set_index("charttime")
    agg = {c: "mean" for c in parametri}
    agg["ossigeno_supplementare"] = "max"
    df = df.groupby(["subject_id", "stay_id"]).resample("10min").agg(agg).dropna(how="all").reset_index()

    # Flag mancanti e forward-fill fino a 4 ore, poi valori fisiologici fissi
    df = df.sort_values(["subject_id", "stay_id", "charttime"])
    for col in parametri:
        df[f"{col}_mancante"] = df[col].isna().astype(int)
    chiavi = [df["subject_id"], df["stay_id"]]
    for col in parametri:
        t_misura = df["charttime"].where(df[col].notna()).groupby(chiavi).ffill()
        riempito = df.groupby(["subject_id", "stay_id"])[col].ffill()
        df[col] = riempito.where(df["charttime"] - t_misura <= pd.Timedelta(hours=4))
    valori_fisiologici = {"freq_respiratoria": 16, "spo2": 97, "pressione_sistolica": 120,
                          "freq_cardiaca": 70, "temperatura": 37.0, "coscienza_gcs": 5}
    for col in parametri:
        df[col] = df[col].fillna(valori_fisiologici[col])

    # NEWS-2 e Delta-NEWS
    df["news2"] = df.apply(calcola_news2, axis=1)
    df = df.sort_values(["subject_id", "stay_id", "charttime"]).set_index("charttime")
    df = (df.groupby(["subject_id", "stay_id"], group_keys=True)
            .apply(features_temporali, include_groups=False).reset_index())
    return df.sort_values(["subject_id", "stay_id", "charttime"]).reset_index(drop=True)

## PARTE 3 — Strada A: dai file CSV originali al formato comune
Stessa logica di estrazione del notebook MIMIC-IV (senza il campionamento: i pazienti sono solo 100):
- parametri vitali per **itemid**, solo righe con valore numerico; °F convertiti in °C
- ossigeno dalle tre fonti: dispositivo indicato = 1 ("None" = 0), FiO2 > 21, flusso > 0

In [4]:
chart = pd.read_csv(CARTELLA_CSV + "icu/chartevents.csv.gz",
                    usecols=["subject_id", "stay_id", "charttime", "itemid", "value", "valuenum"],
                    dtype={"subject_id": "int32", "stay_id": "int32", "itemid": "int32",
                           "value": "string", "valuenum": "float32"})
print(f"chartevents della demo: {len(chart):,} righe, {chart['subject_id'].nunique()} pazienti")

# Parametri vitali
vit_A = chart[chart["itemid"].isin(set(ITEM_IDS) | {ID_TEMP_F}) & chart["valuenum"].notna()].copy()
mask_f = vit_A["itemid"] == ID_TEMP_F
vit_A.loc[mask_f, "valuenum"] = (vit_A.loc[mask_f, "valuenum"] - 32) * 5 / 9   # °F -> °C
vit_A.loc[mask_f, "itemid"] = 223762
vit_A["parametro"] = vit_A["itemid"].map(ITEM_IDS)
vit_A = vit_A[["subject_id", "stay_id", "charttime", "parametro", "valuenum"]]

# Ossigeno
o2 = chart[chart["itemid"].isin(O2_ITEM_IDS)].copy()
o2["_o2"] = np.nan
disp = o2["itemid"] == ID_O2_DISPOSITIVO
o2.loc[disp, "_o2"] = o2.loc[disp, "value"].notna().astype(int)
o2.loc[((o2["itemid"] == ID_O2_FIO2) & (o2["valuenum"] > 21))
       | (o2["itemid"].isin(ID_O2_FLUSSO) & (o2["valuenum"] > 0)), "_o2"] = 1
o2_A = o2.dropna(subset=["_o2"])[["subject_id", "stay_id", "charttime", "_o2"]]

print(f"Strada A — misure dei parametri vitali: {len(vit_A):,} | registrazioni di ossigeno: {len(o2_A):,}")

chartevents della demo: 668,862 righe, 100 pazienti
Strada A — misure dei parametri vitali: 62,274 | registrazioni di ossigeno: 6,111


### Orari validi: il cambio dell'ora legale
MIMIC-IV registra gli orari come ora locale senza fuso orario. Nella prima esecuzione di questa prova le due strade differivano su 2 istanti (8 misure su 62.274): orari come le 02:00 della notte in cui, negli Stati Uniti, gli orologi passano dalle 01:59 alle 03:00 per l'ora legale. Quelle 02:00 non esistono. Lo standard FHIR richiede un istante valido con il fuso orario, quindi la conversione ufficiale le ha spostate alle 03:00.

Per confrontare le due strade sulla stessa base, anche gli orari dei CSV vengono resi validi con la stessa regola: ora locale di New York, e un orario che non esiste viene spostato in avanti all'ora successiva. Gli altri orari non cambiano.

Dettaglio tecnico: gli anni di MIMIC-IV sono spostati nel futuro per l'anonimizzazione (qui 2116 e 2137). Il fuso orario si prende da `zoneinfo`, la libreria standard di Python, che applica le regole dell'ora legale anche a quegli anni; quello usato di default da pandas (`pytz`) le conosce solo fino al 2037 e non sposterebbe nulla.

In [5]:
from zoneinfo import ZoneInfo

def ora_locale_valida(orari):
    # interpreta l'orario come ora locale di New York; un orario inesistente (ora legale) va avanti di un'ora
    t = pd.to_datetime(orari)
    locale = t.dt.tz_localize(ZoneInfo("America/New_York"), nonexistent="shift_forward",
                              ambiguous=np.zeros(len(t), dtype=bool))
    return locale.dt.tz_localize(None)

prima = pd.to_datetime(vit_A["charttime"])
vit_A["charttime"] = ora_locale_valida(vit_A["charttime"])
o2_A["charttime"] = ora_locale_valida(o2_A["charttime"])
print("Misure con l'orario spostato per l'ora legale:", int((prima.values != vit_A["charttime"].values).sum()))

Misure con l'orario spostato per l'ora legale: 8


## PARTE 4 — Strada B: dal FHIR al formato comune
Il lettore FHIR non conosce le tabelle di MIMIC-IV. Per ogni risorsa *Observation* legge:
- il **paziente** e il **ricovero** (riferimenti alle risorse *Patient* ed *Encounter*, da cui si ricavano gli identificativi)
- il **codice**: se è già LOINC lo usa così com'è, se è un codice MIMIC lo traduce con la tabella della Parte 1
- il **valore** (`valueQuantity` per i numeri, `valueString` per il testo) e l'**unità** (°F o °C per la temperatura)
- l'**istante** (`effectiveDateTime`)

Due dettagli del formato FHIR da gestire:
- l'istante ha il fuso orario (es. `2180-07-23T15:00:00-04:00`): si tiene l'ora locale, come nei CSV
- un dispositivo di ossigeno assente è scritto come testo `"None"`: vale 0

In [6]:
def leggi_ndjson(nome_file):
    # un file NDJSON FHIR contiene una risorsa JSON per riga
    with gzip.open(CARTELLA_FHIR + nome_file, "rt", encoding="utf-8") as f:
        for riga in f:
            yield json.loads(riga)

# Identificativi: dalla risorsa FHIR all'id di paziente e di ricovero
paziente_da_ref = {r["id"]: int(r["identifier"][0]["value"]) for r in leggi_ndjson("MimicPatient.ndjson.gz")}
ricovero_da_ref = {r["id"]: int(r["identifier"][0]["value"]) for r in leggi_ndjson("MimicEncounterICU.ndjson.gz")}

def codice_loinc(risorsa):
    # restituisce il codice LOINC della misura, traducendolo se la sorgente usa i codici MIMIC
    for c in risorsa["code"]["coding"]:
        if c["system"] == SISTEMA_LOINC:
            return c["code"]
        if c["system"] == SISTEMA_MIMIC and c["code"] in MIMIC_A_LOINC:
            return MIMIC_A_LOINC[c["code"]]
    return None

righe_vitali, righe_o2 = [], []
for r in leggi_ndjson("MimicObservationChartevents.ndjson.gz"):
    loinc = codice_loinc(r)
    if loinc is None:
        continue                                             # misura che il modello non usa
    base = {"subject_id": paziente_da_ref[r["subject"]["reference"].split("/")[1]],
            "stay_id":    ricovero_da_ref[r["encounter"]["reference"].split("/")[1]],
            "charttime":  r["effectiveDateTime"][:19].replace("T", " ")}   # ora locale, senza fuso
    quantita = r.get("valueQuantity", {})
    valore = np.float32(quantita["value"]) if "value" in quantita else None

    if loinc in LOINC_A_VARIABILE and valore is not None:
        if loinc == "8310-5" and quantita.get("unit") == "°F":
            valore = (valore - np.float32(32)) * np.float32(5) / np.float32(9)   # °F -> °C
        righe_vitali.append({**base, "parametro": LOINC_A_VARIABILE[loinc], "valuenum": valore})
    elif loinc == LOINC_O2_DISPOSITIVO:
        testo = r.get("valueString")
        righe_o2.append({**base, "_o2": 0 if testo in (None, "", "None") else 1})
    elif loinc == LOINC_O2_FIO2 and valore is not None and valore > 21:
        righe_o2.append({**base, "_o2": 1})
    elif loinc == LOINC_O2_FLUSSO and valore is not None and valore > 0:
        righe_o2.append({**base, "_o2": 1})

vit_B = pd.DataFrame(righe_vitali)
o2_B = pd.DataFrame(righe_o2)
print(f"Strada B — misure dei parametri vitali: {len(vit_B):,} | registrazioni di ossigeno: {len(o2_B):,}")

Strada B — misure dei parametri vitali: 62,274 | registrazioni di ossigeno: 6,111


## PARTE 5 — Le due strade producono le 20 feature

In [7]:
feat_A = pipeline_20_feature(vit_A, o2_A)
feat_B = pipeline_20_feature(vit_B, o2_B)
print(f"Strada A: {len(feat_A):,} righe da 10 minuti, {feat_A['subject_id'].nunique()} pazienti, {feat_A['stay_id'].nunique()} ricoveri")
print(f"Strada B: {len(feat_B):,} righe da 10 minuti, {feat_B['subject_id'].nunique()} pazienti, {feat_B['stay_id'].nunique()} ricoveri")

Strada A: 15,184 righe da 10 minuti, 100 pazienti, 140 ricoveri
Strada B: 15,184 righe da 10 minuti, 100 pazienti, 140 ricoveri


## PARTE 6 — Stesso modello sulle due strade
Si usa il modello MIMIC-IV già addestrato (XGBoost) e la sua ricalibrazione isotonica, senza nessun nuovo addestramento.

In [8]:
modello = joblib.load("models/modello_news2_mimic.pkl")
calibratore = joblib.load("models/isotonic_mimic.pkl")
print("Modello:", type(modello).__name__)

for feat in (feat_A, feat_B):
    feat["prob"] = modello.predict_proba(feat[FEATURES])[:, 1]
    feat["prob_calibrata"] = calibratore.predict(feat["prob"])

Modello: XGBClassifier


## PARTE 7 — Confronto
Le righe delle due strade si affiancano per (paziente, ricovero, istante). Per ogni feature e per le probabilità si conta quante righe differiscono e di quanto.

In [9]:
chiave = ["subject_id", "stay_id", "charttime"]
unione = feat_A.merge(feat_B, on=chiave, how="outer", suffixes=("_A", "_B"), indicator=True)
print("Righe presenti in entrambe le strade:", int((unione["_merge"] == "both").sum()))
print("Righe solo nella strada A:", int((unione["_merge"] == "left_only").sum()))
print("Righe solo nella strada B:", int((unione["_merge"] == "right_only").sum()))

entrambe = unione[unione["_merge"] == "both"]
righe = []
for col in FEATURES + ["prob", "prob_calibrata"]:
    diff = (entrambe[f"{col}_A"].astype(float) - entrambe[f"{col}_B"].astype(float)).abs()
    righe.append({"variabile": col, "righe diverse": int((diff > 1e-6).sum()), "differenza massima": float(diff.max())})
confronto = pd.DataFrame(righe)
confronto

Righe presenti in entrambe le strade: 15184
Righe solo nella strada A: 0
Righe solo nella strada B: 0


,variabile,righe diverse,differenza massima
0,freq_respiratoria,0,0.0
1,spo2,0,0.0
2,pressione_sistolica,0,0.0
3,freq_cardiaca,0,0.0
4,temperatura,0,0.0
5,ossigeno_supplementare,0,0.0
6,news2,0,0.0
7,news2_max_6h,0,0.0
8,news2_min_6h,0,0.0
9,news2_media_6h,0,0.0


In [10]:
tutto_uguale = ((unione["_merge"] == "both").all() and (confronto["righe diverse"] == 0).all())
print("=" * 70)
if tutto_uguale:
    print(f"ESITO: le due strade coincidono su tutte le {len(entrambe):,} righe,")
    print("       sulle 20 feature e sulle probabilità (grezze e calibrate).")
else:
    print("ESITO: ci sono differenze tra le due strade: vedi la tabella sopra.")
print("=" * 70)

ESITO: le due strade coincidono su tutte le 15,184 righe,
       sulle 20 feature e sulle probabilità (grezze e calibrate).


## Come si legge l'esito
Se le due strade coincidono, il modello non dipende dal formato di origine: la stessa pipeline, alimentata da dati HL7 FHIR tradotti in LOINC, produce le stesse variabili e le stesse predizioni del formato originale di MIMIC-IV.

**Risultato emerso durante la prova:** l'unica differenza trovata tra le due strade era dovuta al cambio dell'ora legale (orari inesistenti nel formato originale, resi validi dalla conversione in FHIR). È un esempio concreto del valore di uno standard: FHIR obbliga a rappresentare il tempo in modo non ambiguo.

**Limiti della prova:** 100 pazienti della sola MIMIC-IV; la versione FHIR ufficiale usa codici interni di MIMIC per i parametri vitali ICU, quindi la traduzione verso LOINC è stata costruita in questo lavoro; i dati sono file FHIR (NDJSON) e non un server FHIR interrogato in tempo reale; la prova verifica l'equivalenza, non le prestazioni cliniche.